In [1]:
import os, shutil; shutil.copy("/Users/henry.el-jawhari/github/msc_thesis/initialise.py", os.getcwd()); from initialise import *

# Import data
df = pd.read_csv("data/dengue_features_train.csv").merge(pd.read_csv("data/dengue_labels_train.csv"))

df.head()

,city,year,weekofyear,week_start_date,ndvi_ne,ndvi_nw,ndvi_se,ndvi_sw,precipitation_amt_mm,reanalysis_air_temp_k,...,reanalysis_relative_humidity_percent,reanalysis_sat_precip_amt_mm,reanalysis_specific_humidity_g_per_kg,reanalysis_tdtr_k,station_avg_temp_c,station_diur_temp_rng_c,station_max_temp_c,station_min_temp_c,station_precip_mm,total_cases
0,sj,1990,18,1990-04-30,0.122600,0.103725,0.198483,0.177617,12.42,297.572857,...,73.365714,12.42,14.012857,2.628571,25.442857,6.900000,29.4,20.0,16.0,4
1,sj,1990,19,1990-05-07,0.169900,0.142175,0.162357,0.155486,22.82,298.211429,...,77.368571,22.82,15.372857,2.371429,26.714286,6.371429,31.7,22.2,8.6,5
2,sj,1990,20,1990-05-14,0.032250,0.172967,0.157200,0.170843,34.54,298.781429,...,82.052857,34.54,16.848571,2.300000,26.714286,6.485714,32.2,22.8,41.4,4
3,sj,1990,21,1990-05-21,0.128633,0.245067,0.227557,0.235886,15.36,298.987143,...,80.337143,15.36,16.672857,2.428571,27.471429,6.771429,33.3,23.3,4.0,3
4,sj,1990,22,1990-05-28,0.196200,0.262200,0.251200,0.247340,7.52,299.518571,...,80.460000,7.52,17.210000,3.014286,28.942857,9.371429,35.0,23.9,5.8,6


In [2]:
# Predictors and response
y = df["total_cases"]; X = df.drop("total_cases", axis = 1)

In [3]:
# 80:20 train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [4]:
# Pipelines for baseline models, standard scaler, no parameter tuning
rf_pipe     = Pipeline([("encoder", OneHotEncoder(handle_unknown = "ignore")), ("scaler", StandardScaler(with_mean = False)), ("rf", RandomForestRegressor())])
xgb_pipe    = Pipeline([("encoder", OneHotEncoder(handle_unknown = "ignore")), ("scaler", StandardScaler(with_mean = False)), ("xgb", XGBRegressor())])

In [5]:
# Baseline scores
print(f'''
      Baseline scores ---
      Random Forest: {rf_pipe.fit(X_train, y_train).score(X_test, y_test):.3f}
      XGB: {xgb_pipe.fit(X_train, y_train).score(X_test, y_test):.3f}
''')


      Baseline scores ---
      Random Forest: 0.647
      XGB: 0.652



In [7]:
# # Define parameter grid for RF
# rf_param_dist = {
#     "rf__n_estimators": [100, 300, 500],
#     "rf__max_depth": [None, 10, 30],
#     "rf__min_samples_split": [2, 5],
#     "rf__min_samples_leaf": [1, 2],
#     "rf__max_features": ["sqrt", "log2", 1.0],   # 1.0 = all features
#     "rf__bootstrap": [True, False],
#     # Regressor criteria options
#     "rf__criterion": ["squared_error", "absolute_error", "poisson"],  # use "poisson" only if y >= 0
# }

# # Setup RandomizedSearchCV
# rf_random_search = RandomizedSearchCV(
#     estimator = rf_pipe,
#     param_distributions = rf_param_dist,
#     n_iter = 50,
#     scoring = "neg_mean_squared_error",
#     cv = 5,
#     verbose = 1,
#     n_jobs = -1,
#     random_state = 42
# )

# # Fit to training data
# rf_random_search.fit(X_train, y_train)

In [8]:
# # Best score and parameters
# print(f"Best ROC AUC score: {rf_random_search.best_score_:.3f}")
# print("Best parameters:", rf_random_search.best_params_)

In [9]:
# # Define the parameter grid for XGB
# xgb_param_dist = {
#     "xgb__n_estimators": [100, 200, 300, 500],
#     "xgb__max_depth": [3, 5, 7, 10],
#     "xgb__learning_rate": [0.01, 0.05, 0.1, 0.2],
#     "xgb__subsample": [0.6, 0.8, 1.0],
#     "xgb__colsample_bytree": [0.6, 0.8, 1.0],
#     "xgb__gamma": [0, 0.1, 0.3, 0.5],
#     "xgb__reg_alpha": [0, 0.1, 1, 10],
#     "xgb__reg_lambda": [0.1, 1, 10, 100],
#     "xgb__min_child_weight": [1, 3, 5, 10],
#     "xgb__max_delta_step": [0, 1, 5]
# }

# # Setup RandomizedSearchCV
# xgb_random_search = RandomizedSearchCV(
#     estimator = xgb_pipe,
#     param_distributions = xgb_param_dist,
#     n_iter = 50,
#     scoring = "neg_mean_squared_error",
#     cv = 5,
#     verbose = 1,
#     n_jobs = -1,
#     random_state = 42
# )

# # Fit to training data
# xgb_random_search.fit(X_train, y_train)

In [10]:
# # Best score and parameters
# print(f"Best ROC AUC score: {xgb_random_search.best_score_:.3f}")
# print("Best parameters found:", xgb_random_search.best_params_)

In [11]:
# # Pipelines for tuned RF and XGB models
# rf_tuned_pipe   = Pipeline([("scaler", StandardScaler()), ("rf_tuned", RandomForestClassifier(**{x.replace("rf__", ""): v for x, v in rf_random_search.best_params_.items()}))])
# xgb_tuned_pipe  = Pipeline([("scaler", StandardScaler()), ("xgb_tuned", XGBClassifier(**{x.replace("xgb__", ""): v for x, v in xgb_random_search.best_params_.items()}))])

In [12]:
# predictions = (
#     pd.concat(
#         [
#             pd.DataFrame({"actual": list(y_test), "predicted": rf_tuned_pipe.fit(X_train, y_train).predict_proba(X_test)[:,1]}),
#             pd.DataFrame({"actual": list(y_test), "predicted": xgb_tuned_pipe.fit(X_train, y_train).predict_proba(X_test)[:,1]}),
#         ], 
#         keys = ["rf", "xgb"]
#         )
#         .reset_index(names = ["model", "index"])
#         .drop("index", axis = 1)
#         )

# predictions.head()

In [13]:
# # Save data and the hyper-parameter tuned RF and XGB models
# modelling_output = {
#     "X_train": X_train,
#     "y_train": y_train,
#     "X_test": X_test,
#     "y_test": y_test,
#     "rf_model": rf_tuned_pipe,
#     "xgb_model": xgb_tuned_pipe,
#     "predictions": predictions
# }

# with open("modelling_output.pickle", "wb") as file:
#     pickle.dump(modelling_output, file, protocol = pickle.HIGHEST_PROTOCOL)